# Clipper Whisper Worker (Colab GPU)
Jalankan semua cell berurutan. Di cell terakhir, copy URL tunnel + token ke Settings Clipper.


In [ ]:
!pip install -q "faster-whisper==1.1.1" "av>=12,<19"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import av; print('av version:', av.__version__)
assert int(av.__version__.split('.')[0]) < 19, 'PyAV 19+ tidak kompatibel dengan faster-whisper 1.1.1; jalankan ulang install cell'


In [ ]:
import os, secrets
os.environ.setdefault('WORKER_TOKEN', os.getenv('WORKER_TOKEN') or secrets.token_urlsafe(24))
os.environ.setdefault('WHISPER_MODEL', 'large-v3-turbo')
os.environ.setdefault('WHISPER_COMPUTE', 'float16')
os.environ.setdefault('ALLOW_CPU_FALLBACK', '1')
open('/content/.clipper_token','w').write(os.environ['WORKER_TOKEN'])
import site, glob
libdirs = []
for sp in site.getsitepackages():
    libdirs += glob.glob(sp + '/nvidia/*/lib')
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs + [os.environ.get('LD_LIBRARY_PATH','')])
print('='*60)
print('WORKER_TOKEN :', os.environ['WORKER_TOKEN'])
print('MODEL        :', os.environ['WHISPER_MODEL'])
print('av           :', __import__('av').__version__)
print('='*60)


In [ ]:
# Hotfix kompatibilitas: kalau Colab memaksa PyAV 19+, buang kwarg yang sudah dihapus.
import av, inspect
if 'metadata_errors' not in inspect.signature(av.open).parameters:
    _orig_open = av.open
    def _open_compat(*a, **kw):
        for key in ('metadata_errors','metadata_encoding','stream_options'):
            kw.pop(key, None)
        return _orig_open(*a, **kw)
    av.open = _open_compat
print('av patch applied (metadata_errors):', 'metadata_errors' not in inspect.signature(av.open).parameters)


In [ ]:
# Opsional: login HF untuk download lebih cepat (hindari rate limit).
# from huggingface_hub import login; login()
print('HF login dilewati (opsional).')


In [ ]:
!wget -q https://raw.githubusercontent.com/Maulana2505/clipper-Remotion-renderer-/main/colab/worker.py -O /content/worker.py
import os
print('worker.py ready:', os.path.exists('/content/worker.py'))


In [ ]:
# Verifikasi CUDA + model + decode audio sebelum tunnel dijalankan
import os, ctranslate2, av, inspect
print('cuda devices:', ctranslate2.get_cuda_device_count())
assert int(av.__version__.split('.')[0]) < 19, 'PyAV 19+ tidak kompatibel; jalankan ulang install cell'
from faster_whisper import WhisperModel
m = WhisperModel(os.environ['WHISPER_MODEL'], device='cuda', compute_type=os.environ['WHISPER_COMPUTE'])
print('model loaded on cuda OK')
import tempfile, wave
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
    w=wave.open(f.name,'wb'); w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(b'\x00\x00'*48000); w.close(); p=f.name
from faster_whisper.audio import decode_audio
audio = decode_audio(p, sampling_rate=16000)
print('decode ok, samples:', len(audio))
segs,_=m.transcribe(p, language='en'); print('smoke segments:', len(list(segs)))
import os; os.remove(p)


In [ ]:
# Tampilkan kredensial kapan saja (jalankan ulang cell ini)
import os
tok = open('/content/.clipper_token').read().strip() if os.path.exists('/content/.clipper_token') else '(belum ada)'
url = open('/content/.clipper_url').read().strip() if os.path.exists('/content/.clipper_url') else '(belum ada - jalankan cell tunnel)'
print('='*60)
print('TUNNEL URL :', url)
print('TOKEN      :', tok)
print('='*60)


In [ ]:
import subprocess, time, re, threading
def run_worker():
    subprocess.run(['uvicorn','worker:app','--host','0.0.0.0','--port','8010','--workers','1'])
threading.Thread(target=run_worker, daemon=True).start()
time.sleep(12)
import urllib.request
print(urllib.request.urlopen('http://127.0.0.1:8010/health').read().decode())
print('TOKEN      :', open('/content/.clipper_token').read().strip())
proc = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8010'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for i,line in enumerate(proc.stdout):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        open('/content/.clipper_url','w').write(url)
        break
    if i > 2000:
        break
if url:
    print('='*60)
    print('TUNNEL URL :', url)
    print('TOKEN      :', open('/content/.clipper_token').read().strip())
    print('='*60)
else:
    print('URL tunnel tidak ditemukan. Cek output cloudflared di atas.')
    print('TOKEN (tetap bisa dipakai kalau URL sudah ada):', open('/content/.clipper_token').read().strip())
proc.wait()
